### Objetivo y alcance

Demostración académica de mantenimiento de un pipeline tabular. El objetivo y los
cambios de distribución son sintéticos; los siete vitales proceden de PhysioNet.
Las cifras y figuras se calculan, nunca se prescriben. Ejecutar de arriba abajo.

### Bibliotecas, rutas y parámetros reproducibles



In [ ]:
import os, json, time, copy, random, traceback, hashlib, platform
from pathlib import Path
from collections import deque
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import xgboost, sklearn
import torch
from torch import nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
WEIGHTS = np.array([1.2, -1.1, .3, -.2, -.7, .2, .8], dtype=np.float32)
SCENARIOS = ('stable', 'covariate', 'concept', 'combined')
ACTIONS = ('A1', 'A2', 'A3', 'A4')
SMOKE = os.environ.get('DARL_DUMMY_SMOKE') == '1'
DAYS = 4 if SMOKE else 16
EPISODES = 2 if SMOKE else 200
OUT = Path(os.environ.get('DARL_DUMMY_OUTPUT', '/kaggle/working' if Path('/kaggle/input').exists() else 'kaggle/dqn-dummy/v1/outputs'))
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'figures').mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
progress = {'status': 'running', 'stage': 'setup', 'seed': SEED, 'device': str(DEVICE), 'smoke': SMOKE,
            'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                         'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__, 'torch': torch.__version__}}
def checkpoint(stage, **extra):
    """Persist a compact progress record after each costly stage."""
    progress.update(stage=stage, **extra)
    def clean(value):
        if isinstance(value, dict):return {str(k):clean(v) for k,v in value.items()}
        if isinstance(value, (list,tuple)):return [clean(v) for v in value]
        if isinstance(value,(float,np.floating)) and not np.isfinite(value):return None
        if isinstance(value,np.generic):return value.item()
        return value
    (OUT/'run_summary.json').write_text(json.dumps(clean(progress), indent=2, default=str, allow_nan=False), encoding='utf-8')
checkpoint('setup')


### Cargar columnas autorizadas y verificar llaves temporales



In [ ]:
def find_parquet():
    """Resolve the one named Kaggle dataset or an explicit local test path."""
    custom = os.environ.get('DARL_DUMMY_PARQUET')
    candidates = [Path(custom)] if custom else list(Path('/kaggle/input').rglob('physionet.parquet'))
    candidates += [Path('data/processed/physionet/physionet.parquet')]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    mounts=sorted(p.name for p in Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else []
    raise FileNotFoundError(f'physionet.parquet unavailable; mounted inputs={mounts}')

DATA_PATH = find_parquet()
frame = pd.read_parquet(DATA_PATH, columns=['patient_id','source_set','ICULOS','SepsisLabel',*VITALS])
assert not frame.duplicated(['patient_id','ICULOS']).any()
assert set(frame.source_set.unique()) == {'a','b'}
frame = frame.sort_values(['patient_id','ICULOS']).reset_index(drop=True)
assert frame.groupby('patient_id').ICULOS.apply(lambda v: v.is_monotonic_increasing).all()
# ffill causal, sin mezclar pacientes; conservar los NaN originales para la inyección
frame[VITALS] = frame.groupby('patient_id', sort=False)[VITALS].ffill()
checkpoint('loaded', n_rows=len(frame), n_patients=int(frame.patient_id.nunique()))
print('Filas:', len(frame), '| pacientes:', frame.patient_id.nunique())


### Particiones por paciente y etiqueta artificial independiente



In [ ]:
patient_labels = frame.groupby(['source_set','patient_id'], sort=False).SepsisLabel.max().reset_index()
def split_ids(site, fractions):
    """Make deterministic patient-disjoint stratified partitions."""
    p = patient_labels.loc[patient_labels.source_set.eq(site)]
    ids, held = train_test_split(p.patient_id.to_numpy(), test_size=1-fractions[0], stratify=p.SepsisLabel, random_state=SEED)
    labels = p.set_index('patient_id').loc[held, 'SepsisLabel'].to_numpy()
    val, test = train_test_split(held, test_size=fractions[2]/(fractions[1]+fractions[2]), stratify=labels, random_state=SEED)
    return ids, val, test

a_train_ids, a_val_ids, a_test_ids = split_ids('a', (.70,.15,.15))
b_train_ids, b_val_ids, b_test_ids = split_ids('b', (.60,.20,.20))
all_parts = [set(x) for x in (a_train_ids,a_val_ids,a_test_ids,b_train_ids,b_val_ids,b_test_ids)]
assert all(not (all_parts[i] & all_parts[j]) for i in range(6) for j in range(i+1,6))

def roles(ids):
    """Partition B patients into update and held-out evaluation roles."""
    p = patient_labels.set_index('patient_id').loc[ids]
    u, e = train_test_split(ids, test_size=.5, stratify=p.SepsisLabel, random_state=SEED)
    assert not set(u) & set(e)
    return sorted(u), sorted(e)

B_PARTS = {'train':roles(b_train_ids), 'val':roles(b_val_ids), 'test':roles(b_test_ids)}
a_train = frame.loc[frame.patient_id.isin(a_train_ids)]
REF_MED = a_train[VITALS].median().to_numpy(dtype=np.float32)
REF_IQR = (a_train[VITALS].quantile(.75)-a_train[VITALS].quantile(.25)).clip(lower=1e-6).to_numpy(dtype=np.float32)
def canonical_scores(rows):
    """Compute the synthetic target on causal vitals before measurement change."""
    raw = rows[VITALS].to_numpy(dtype=np.float32)
    raw = np.where(np.isfinite(raw), raw, REF_MED)
    return (np.clip((raw-REF_MED)/REF_IQR, -4, 4)*WEIGHTS).sum(axis=1)
THRESHOLD = float(np.median(canonical_scores(a_train)))
scores = canonical_scores(frame)
# Random flip is stable for a row regardless of replay order or partition.
def flip_bit(pid, hour):
    digest = hashlib.blake2b(f'{SEED}:{pid}:{hour}'.encode(), digest_size=8).digest()
    return int.from_bytes(digest, 'big') / 2**64 < .05
flips = np.fromiter((flip_bit(p,h) for p,h in zip(frame.patient_id,frame.ICULOS)), dtype=bool, count=len(frame))
frame['y_dummy'] = np.logical_xor(scores > THRESHOLD, flips).astype(np.int8)
assert frame.y_dummy.nunique() == 2
a_train = frame.loc[frame.patient_id.isin(a_train_ids)]
checkpoint('partitioned', partitions={'a_train':len(a_train_ids),'a_val':len(a_val_ids),'a_test':len(a_test_ids),
                            'b_train':len(b_train_ids),'b_val':len(b_val_ids),'b_test':len(b_test_ids)},
           dummy_prevalence=float(frame.y_dummy.mean()), target_threshold=THRESHOLD)
print('Particiones disjuntas; prevalencia dummy:', round(frame.y_dummy.mean(),3))


### Replay de 200 camas con recambio horario y separación de roles



In [ ]:
PATIENT_LENGTHS = frame.groupby('patient_id',sort=False).size().to_dict()
PATIENT_TABLE = frame.assign(position=frame.groupby('patient_id',sort=False).cumcount()).set_index(['patient_id','position'])
def make_replay(parts, days=DAYS, seed=SEED):
    """Replay observed rows once per patient; replace exhausted beds next hour."""
    rng = np.random.default_rng(seed)
    selected = []
    for role, ids in zip(('update','eval'), parts):
        queue = list(rng.permutation(ids))
        beds = [(queue.pop(),0) for _ in range(100)]
        for hour in range(days*24):
            for bed in range(100):
                pid,pos = beds[bed]
                selected.append((pid,pos,role,bed+(0 if role=='update' else 100),hour))
                pos += 1
                if pos == PATIENT_LENGTHS[pid]:
                    if hour < days*24-1:
                        if not queue: raise RuntimeError(f'Patient queue exhausted: {role}, hour {hour}')
                        beds[bed] = (queue.pop(),0)
                else:
                    beds[bed] = (pid,pos)
    keys = pd.DataFrame(selected,columns=['patient_id','position','role','bed','hour'])
    # Per-patient position indexes the sorted, causally filled observation.
    lookup = PATIENT_TABLE.reindex(pd.MultiIndex.from_frame(keys[['patient_id','position']])).reset_index()
    schedule = pd.concat([keys[['role','bed','hour']],lookup],axis=1)
    schedule['day'] = schedule.hour // 24 + 1
    assert len(schedule)==days*24*200
    assert schedule.groupby('hour').size().eq(200).all()
    assert schedule.groupby(['hour','role']).size().eq(100).all()
    return schedule


### Pipeline selectivo: mediana/IQR, clipping y XGBoost



In [ ]:
class Pipeline:
    """Seven-vital transform and classifier with separately replaceable stages."""
    def __init__(self, med, iqr, model):
        self.med = np.asarray(med,dtype=np.float32).copy()
        self.iqr = np.asarray(iqr,dtype=np.float32).copy()
        self.model = model
    def transform(self, rows):
        """Apply current imputation, robust scaling and explicit clipping."""
        x=rows[VITALS].to_numpy(dtype=np.float32)
        x=np.where(np.isfinite(x),x,self.med)
        z=(x-self.med)/np.maximum(self.iqr,1e-6)
        return np.clip(z,-2,2)
    def predict(self, rows):
        """Return positive-class probability for observed rows."""
        return self.model.predict_proba(self.transform(rows))[:,1]
    def saturation(self, rows):
        """Fraction of feature cells at a clipping boundary."""
        x=rows[VITALS].to_numpy(dtype=np.float32)
        z=(np.where(np.isfinite(x),x,self.med)-self.med)/np.maximum(self.iqr,1e-6)
        return float(np.mean(np.abs(z)>2))
    def act(self, action, rows):
        """Return a new pipeline and measured action time without mutating self."""
        assert 0<=action<4
        start=time.perf_counter()
        if action==0:
            return self, time.perf_counter()-start
        med,iqr=self.med,self.iqr
        if action in (1,3):
            med=rows[VITALS].median().fillna(pd.Series(self.med,index=VITALS)).to_numpy(dtype=np.float32)
            q=rows[VITALS].quantile(.75)-rows[VITALS].quantile(.25)
            iqr=q.fillna(pd.Series(self.iqr,index=VITALS)).clip(lower=1e-6).to_numpy(dtype=np.float32)
        model=self.model
        if action in (2,3):
            if rows.y_dummy.nunique()<2: raise ValueError('Update needs both dummy classes')
            x=Pipeline(med,iqr,self.model).transform(rows)
            model=XGBClassifier(n_estimators=60,max_depth=3,learning_rate=.1,tree_method='hist',
                                device='cpu',n_jobs=2,random_state=SEED,eval_metric='logloss')
            model.fit(x,rows.y_dummy.to_numpy())
        new=Pipeline(med,iqr,model)
        return new,time.perf_counter()-start

initial_model=XGBClassifier(n_estimators=60,max_depth=3,learning_rate=.1,tree_method='hist',
                            device='cpu',n_jobs=2,random_state=SEED,eval_metric='logloss')
initial_model.fit(Pipeline(REF_MED,REF_IQR,None).transform(a_train),a_train.y_dummy.to_numpy())
INITIAL=Pipeline(REF_MED,REF_IQR,initial_model)
probe_rows=a_train.iloc[:2400]
assert probe_rows.y_dummy.nunique()==2
for probe_action in (0,1,2,3):
    probe_pipe,probe_seconds=INITIAL.act(probe_action,probe_rows)
    assert probe_seconds>=0
    assert (probe_pipe.model is INITIAL.model)==(probe_action in (0,1))
    if probe_action in (0,2):assert np.array_equal(probe_pipe.med,INITIAL.med)
del probe_pipe,probe_rows
def auc(rows,pipe):
    """Return NaN for a one-class evaluation day."""
    y=rows.y_dummy.to_numpy()
    return float(roc_auc_score(y,pipe.predict(rows))) if np.unique(y).size==2 else np.nan
held_a=frame.loc[frame.patient_id.isin(a_test_ids)]
a_test_auc=auc(held_a,INITIAL)
checkpoint('initial_model', a_test_auc=a_test_auc)
print('A-test AUC dummy:',round(a_test_auc,4))


### Cambios estadísticos controlados sobre copias del replay



In [ ]:
def inject(schedule,scenario,offset=4.0,scale=1.4,onset=5):
    """Apply measured numeric stress while preserving missingness and provenance."""
    assert scenario in SCENARIOS
    out=schedule.copy()
    active=out.day.ge(onset)
    if scenario in ('covariate','combined'):
        # Different signs keep drift visible on all seven vitals.
        direction=np.array([1,-1,1,1,-1,1,1],dtype=np.float32)
        src=out.loc[active,VITALS].to_numpy(dtype=np.float32)
        changed=REF_MED+scale*(src-REF_MED)+offset*REF_IQR*direction
        out.loc[active,VITALS]=changed
    if scenario in ('concept','combined'):
        out.loc[active,'y_dummy']=1-out.loc[active,'y_dummy']
    out['scenario']=scenario
    out['drift_active']=active & (scenario!='stable')
    return out


### KS, PSI y AUC antes de actuar; referencia congelada de B-train



In [ ]:
reference_replay=make_replay(B_PARTS['train'],days=4,seed=SEED)
REFERENCE={}
for col in VITALS:
    clean=reference_replay[col].dropna().to_numpy(dtype=float)
    edges=np.unique(np.quantile(clean,np.linspace(0,1,11)))
    edges[0],edges[-1]=-np.inf,np.inf
    REFERENCE[col]={'sample':clean[::max(1,len(clean)//1000)][:1000],'edges':edges,
                    'hist':np.histogram(clean,bins=edges)[0]}
def psi(reference,current):
    """Population stability index on frozen reference bins."""
    edges=reference['edges']
    if len(edges)<2: return 0.0
    q=np.histogram(np.asarray(current,dtype=float),bins=edges)[0].astype(float)+1e-6
    p=reference['hist'].astype(float)+1e-6
    p/=p.sum();q/=q.sum()
    return float(np.sum((q-p)*np.log(q/p)))
def monitor(rows,pipe):
    """Return maximum per-vital KS/PSI and the day's held-out AUC."""
    ks_values=[];psi_values=[]
    for col in VITALS:
        now=rows[col].dropna().to_numpy(dtype=float)
        if len(now)==0: continue
        ref=REFERENCE[col]
        ks_values.append(float(ks_2samp(ref['sample'],now,method='asymp').statistic))
        psi_values.append(psi(ref,now))
    eval_rows=rows.loc[rows.role.eq('eval')]
    return {'ks':max(ks_values,default=0.),'psi':max(psi_values,default=0.),
            'auc':auc(eval_rows,pipe),'prevalence':float(eval_rows.y_dummy.mean()),
            'saturation':pipe.saturation(rows)}
def encode_observation(metrics,last_auc=.5):
    """Encode three thresholded metrics and an AUC-validity bit."""
    value=metrics['auc'] if np.isfinite(metrics['auc']) else last_auc
    bins=[np.searchsorted([.10,.30],metrics['ks'],side='right'),
          np.searchsorted([.10,.25],metrics['psi'],side='right'),
          np.searchsorted([.65,.85],value,side='right')]
    result=np.zeros(10,dtype=np.float32)
    for i,b in enumerate(bins): result[i*3+b]=1
    result[9]=float(np.isfinite(metrics['auc']))
    return result


### Entorno con recompensa observable al cierre del día siguiente



In [ ]:
class BedEnv:
    """Partially observed daily maintenance environment over one replay."""
    def __init__(self,schedule,initial=INITIAL):
        self.schedule=schedule
        self.initial=initial
        self.days=int(schedule.day.max())
        self.parts={d:schedule.loc[schedule.day.eq(d)] for d in range(1,self.days+1)}
        self.reset()
    def reset(self):
        """Restore initial pipeline and return the first three-day history."""
        self.pipe=self.initial
        self.day=1
        self.history=deque(maxlen=3)
        self.last_auc=.5
        self.actions=[];self.daily=[]
        self.current=self._observe(1)
        return self.current[0],{}
    def _observe(self,day):
        measured=monitor(self.parts[day],self.pipe)
        block=encode_observation(measured,self.last_auc)
        if np.isfinite(measured['auc']):self.last_auc=measured['auc']
        self.history.append(block)
        vector=np.concatenate([np.zeros(10,dtype=np.float32)]*(3-len(self.history))+list(self.history))
        return vector,measured
    def step(self,action):
        """Act at day close and reveal next-day held-out utility."""
        if self.day>=self.days:raise RuntimeError('Episode already finished')
        before=self.current[1]
        day=self.day
        update=self.parts[day].loc[self.parts[day].role.eq('update')]
        old=self.pipe
        try:
            changed,elapsed=old.act(int(action),update)
            executed=int(action);reason=''
        except ValueError as exc:
            changed,elapsed=old.act(0,update)
            executed=0;reason=str(exc)
        self.pipe=changed
        self.day+=1
        self.current=self._observe(self.day)
        next_state,after=self.current
        valid=bool(np.isfinite(after['auc']))
        reward=float(after['auc']-.1*elapsed) if valid else np.nan
        done=self.day==self.days
        record={'day':day,'scenario':str(self.schedule.scenario.iloc[0]),'requested':int(action),
                'executed':executed,'fallback_reason':reason,'time_s':float(elapsed),
                'auc_before':before['auc'],'auc_next':after['auc'],'reward':reward,
                'reward_valid':valid,'drift_active':bool(self.parts[day].drift_active.iloc[0]),
                'next_drift_active':bool(self.parts[self.day].drift_active.iloc[0]),
                'ks':before['ks'],'psi':before['psi'],'saturation':before['saturation'],
                'saturation_next':after['saturation']}
        self.actions.append(record)
        self.daily.append({'day':day,'scenario':record['scenario'],**before,'action':ACTIONS[executed],
                           'reward':reward,'time_s':float(elapsed),'drift_active':record['drift_active']})
        if done:
            self.daily.append({'day':self.day,'scenario':record['scenario'],**after,'action':'—',
                               'reward':np.nan,'time_s':0.,'drift_active':record['next_drift_active']})
        return next_state,reward,done,False,{'record':record,'reward_valid':valid,'next_metrics':after}


### DQN de clase con replay FIFO y red target congelada



In [ ]:
class QNetwork(nn.Module):
    """Estimate four action values from the last three daily observations."""
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(30,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,4))
    def forward(self,x):
        """Return unconstrained Q values."""
        return self.net(x)

class ReplayBuffer:
    """FIFO transitions with uniform minibatch sampling."""
    def __init__(self,capacity=50000):
        self.buffer=deque(maxlen=capacity)
        self.rng=random.Random(SEED)
    def push(self,s,a,r,s2,done):
        """Store a complete observed transition."""
        self.buffer.append((s.copy(),int(a),float(r),s2.copy(),bool(done)))
    def sample(self,n):
        """Return a uniform minibatch."""
        batch=self.rng.sample(self.buffer,n)
        s,a,r,s2,d=zip(*batch)
        return (np.stack(s),np.array(a),np.array(r,dtype=np.float32),np.stack(s2),np.array(d,dtype=np.float32))
    def __len__(self):
        """Return current capacity use."""
        return len(self.buffer)

class DQNAgent:
    """Epsilon-greedy DQN with hard target copies every 50 updates.

    Bellman values center reward by a constant for numerical learning; all
    episodes have the same horizon, so this action-independent shift preserves
    action ordering. CSV and replay retain the specified uncentered reward.
    """
    def __init__(self):
        self.online=QNetwork().to(DEVICE)
        self.target=QNetwork().to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.target.requires_grad_(False)
        self.optim=torch.optim.Adam(self.online.parameters(),lr=.001)
        self.replay=ReplayBuffer()
        self.rng=np.random.default_rng(SEED)
        self.epsilon=1.
        self.updates=0
        self.losses=[]
    def select(self,obs,greedy=False):
        """Choose random exploration or highest online Q value."""
        if not greedy and self.rng.random()<self.epsilon:return int(self.rng.integers(4))
        with torch.no_grad():
            s=torch.as_tensor(obs,dtype=torch.float32,device=DEVICE).unsqueeze(0)
            return int(self.online(s).argmax(1).item())
    def learn(self,s,a,r,s2,done):
        """Add valid delayed outcome and make one Bellman gradient step."""
        if not np.isfinite(r):return None
        self.replay.push(s,a,r,s2,done)
        if len(self.replay)<64:return None
        states,actions,rewards,next_states,terminals=self.replay.sample(64)
        states=torch.as_tensor(states,dtype=torch.float32,device=DEVICE)
        actions=torch.as_tensor(actions,dtype=torch.long,device=DEVICE).unsqueeze(1)
        rewards=torch.as_tensor(rewards,dtype=torch.float32,device=DEVICE)
        following=torch.as_tensor(next_states,dtype=torch.float32,device=DEVICE)
        terminal=torch.as_tensor(terminals,dtype=torch.float32,device=DEVICE)
        values=self.online(states).gather(1,actions).squeeze(1)
        with torch.no_grad():targets=(rewards-.8)+.99*(1-terminal)*self.target(following).max(1).values
        loss=nn.functional.mse_loss(values,targets)
        self.optim.zero_grad();loss.backward()
        torch.nn.utils.clip_grad_value_(self.online.parameters(),10.)
        self.optim.step()
        self.updates+=1
        self.epsilon=max(.01,self.epsilon*.995)
        if self.updates%50==0:self.target.load_state_dict(self.online.state_dict())
        self.losses.append(float(loss.item()))
        return float(loss.item())


### Contratos básicos antes del entrenamiento largo



In [ ]:
assert len(encode_observation({'ks':0.,'psi':0.,'auc':.9}))==10
small_buffer=ReplayBuffer(2)
for i in range(3):small_buffer.push(np.zeros(30,dtype=np.float32),0,.5,np.zeros(30,dtype=np.float32),False)
assert len(small_buffer)==2
temp_agent=DQNAgent()
assert all(not p.requires_grad for p in temp_agent.target.parameters())
assert len({id(p) for g in temp_agent.optim.param_groups for p in g['params']} & {id(p) for p in temp_agent.target.parameters()})==0
del temp_agent,small_buffer


### Calibrar severidad solo con B-train y congelarla antes de prueba



In [ ]:
def calibrate():
    """Select least severe candidate with measurable held-out degradation and repair."""
    base=make_replay(B_PARTS['train'],days=max(7,DAYS),seed=SEED)
    clean_next=base.loc[base.day.eq(6) & base.role.eq('eval')]
    stable_auc=auc(clean_next,INITIAL)
    records=[]
    for offset in (3.,4.,5.):
        for scale in (1.4,1.8):
            row={'offset':offset,'scale':scale,'stable_auc':stable_auc}
            for scenario,action in (('covariate',1),('combined',3)):
                shifted=inject(base,scenario,offset,scale,onset=5)
                day5=shifted.loc[shifted.day.eq(5)]
                day6=shifted.loc[shifted.day.eq(6) & shifted.role.eq('eval')]
                before=auc(day6,INITIAL)
                adapted,_=INITIAL.act(action,day5.loc[day5.role.eq('update')])
                after=auc(day6,adapted)
                row[f'{scenario}_before']=before;row[f'{scenario}_after']=after
                row[f'{scenario}_gain']=after-before
            records.append(row)
    table=pd.DataFrame(records)
    table.to_csv(OUT/'calibration.csv',index=False)
    qualified=table.loc[(table.covariate_gain>.03)&(table.combined_gain>.03)&
                        ((table.stable_auc-table.covariate_before)>.03)&
                        ((table.stable_auc-table.combined_before)>.03)]
    choice=(qualified if len(qualified) else table.sort_values(['covariate_gain','combined_gain'],ascending=False)).iloc[0]
    return float(choice.offset),float(choice.scale),bool(len(qualified)),table


### Reproducir una política sin acceso al escenario durante decisión



In [ ]:
def choose_policy(policy,agent,obs,metrics,rng):
    """Choose among DQN, four fixed actions, random and fixed thresholds."""
    if policy=='dqn':return agent.select(obs,greedy=True)
    if policy in ACTIONS:return ACTIONS.index(policy)
    if policy=='random':return int(rng.integers(4))
    if policy=='heuristic':
        cov=metrics['ks']>=.30 or metrics['psi']>=.25
        low=np.isfinite(metrics['auc']) and metrics['auc']<.85
        return 3 if cov and low else 1 if cov else 2 if low else 0
    raise ValueError(policy)

def roll(schedule,policy,agent=None,audit=False):
    """Return auditable action/day records and optional one-step action oracle."""
    env=BedEnv(schedule)
    obs,_=env.reset()
    rng=np.random.default_rng(SEED)
    audit_rows=[]
    while env.day<env.days:
        day=env.day
        metrics=env.current[1]
        action=choose_policy(policy,agent,obs,metrics,rng)
        if audit:
            update=env.parts[day].loc[env.parts[day].role.eq('update')]
            next_eval=env.parts[day+1].loc[env.parts[day+1].role.eq('eval')]
            utilities=[]
            for candidate in range(4):
                try:
                    new,elapsed=env.pipe.act(candidate,update)
                    candidate_auc=auc(next_eval,new)
                    utilities.append(candidate_auc-.1*elapsed if np.isfinite(candidate_auc) else np.nan)
                except ValueError:utilities.append(np.nan)
        obs,reward,done,_,info=env.step(action)
        if audit:
            chosen=info['record']
            utilities[chosen['executed']]=chosen['reward']
            best=float(np.nanmax(utilities)) if np.isfinite(utilities).any() else np.nan
            audit_rows.append({'scenario':chosen['scenario'],'day':day,'chosen':ACTIONS[chosen['executed']],
                               'frontier':bool(chosen['drift_active']!=chosen['next_drift_active']),
                               **{f'utility_{ACTIONS[i]}':utilities[i] for i in range(4)},
                               'chosen_utility':chosen['reward'],'oracle_utility':best,
                               'regret':best-chosen['reward'] if np.isfinite(best) and np.isfinite(chosen['reward']) else np.nan})
        if done:break
    actions=pd.DataFrame(env.actions).assign(policy=policy)
    daily=pd.DataFrame(env.daily).assign(policy=policy)
    valid=actions.reward.dropna()
    result={'scenario':schedule.scenario.iloc[0],'policy':policy,
            'mean_auc':float(daily.auc.mean()),'return':float(valid.sum()),
            'mean_reward':float(valid.mean()) if len(valid) else np.nan,
            'seconds':float(actions.time_s.sum()),'n_valid':int(len(valid)),
            **{f'count_{name}':int(actions.executed.eq(i).sum()) for i,name in enumerate(ACTIONS)}}
    return result,actions,daily,pd.DataFrame(audit_rows)


### Entrenar DQN y elegir checkpoint solo con validación



In [ ]:
def train_agent(offset,scale):
    """Run balanced scenario episodes and frozen validation checkpoints."""
    agent=DQNAgent()
    train_replays=[make_replay(B_PARTS['train'],days=DAYS,seed=SEED+i) for i in range(4)]
    val_base=make_replay(B_PARTS['val'],days=DAYS,seed=SEED)
    history=[];checkpoints=[]
    max_eps=4 if SMOKE else 400
    minimum=4 if SMOKE else 200
    for ep in range(1,max_eps+1):
        scenario=SCENARIOS[(ep-1)%4]
        onset=5+(ep%4) if DAYS>=8 else 2
        episode=inject(train_replays[(ep-1)%4],scenario,offset,scale,onset=onset)
        env=BedEnv(episode)
        obs,_=env.reset()
        episode_return=0.
        while env.day<env.days:
            action=agent.select(obs)
            next_obs,reward,done,_,info=env.step(action)
            if info['reward_valid']:
                agent.learn(obs,action,reward,next_obs,done)
                episode_return+=reward
            obs=next_obs
            if done:break
        row={'episode':ep,'scenario':scenario,'train_return':episode_return,
             'epsilon':agent.epsilon,'loss':agent.losses[-1] if agent.losses else np.nan,
             'updates':agent.updates,'validation_return':np.nan}
        if ep%10==0 or ep==max_eps or (SMOKE and ep==minimum):
            vals=[];signatures=[]
            for name in SCENARIOS:
                val=inject(val_base,name,offset,scale,onset=5 if DAYS>=5 else 2)
                result,actions,_,_=roll(val,'dqn',agent)
                vals.append(result['return']);signatures.extend(actions.executed.to_list())
            row['validation_return']=float(np.mean(vals))
            checkpoints.append({'episode':ep,'return':row['validation_return'],
                                'weights':copy.deepcopy(agent.online.state_dict()),'signature':signatures})
            pd.DataFrame(history+[row]).to_csv(OUT/'training_history.csv',index=False)
            checkpoint('training',episodes_done=ep,updates=agent.updates,best_validation_return=max(c['return'] for c in checkpoints))
        history.append(row)
        if ep>=minimum and len(checkpoints)>=5:
            latest=checkpoints[-5:]
            scores=np.array([item['return'] for item in latest])
            spread=float(np.ptp(scores)/max(abs(scores.mean()),1e-6))
            same=float(np.mean(np.array(latest[-1]['signature'])==np.array(latest[-2]['signature'])))
            if spread<=.05 and same>=.95:break
    winner=max(checkpoints,key=lambda item:item['return'])
    agent.online.load_state_dict(winner['weights'])
    pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
    spread=np.array([x['return'] for x in checkpoints[-5:]],dtype=float)
    variation=float(np.ptp(spread)/max(abs(spread.mean()),1e-6)) if len(spread)==5 else np.nan
    stability=float(np.mean(np.array(checkpoints[-1]['signature'])==np.array(checkpoints[-2]['signature']))) if len(checkpoints)>1 else np.nan
    return agent,history,checkpoints,{'selected_episode':winner['episode'],'updates':agent.updates,
                                    'episodes':len(history),'validation_variation':variation,
                                    'policy_stability':stability}


### Ocho figuras de ocupación, cambio, entrenamiento y decisiones



In [ ]:
def make_figures(stable_schedule,train_history,daily,actions,audit,comparison):
    """Save eight 16:9 PNG/SVG figures from observed, auditable tables."""
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    files=[]
    def save(fig,name):
        fig.tight_layout()
        for ext in ('png','svg'):
            path=OUT/'figures'/f'{name}.{ext}'
            fig.savefig(path,dpi=160 if ext=='png' else None,bbox_inches='tight')
            files.append(str(path.relative_to(OUT)))
        plt.close(fig)
    fig,ax=plt.subplots(2,1,figsize=(16,9),sharex=True)
    occ=stable_schedule.groupby('hour').bed.nunique()
    ax[0].plot(occ.index/24+1,occ.values,color='#245c78')
    ax[0].set(ylabel='Camas ocupadas',ylim=(0,220),title='Ocupacion del replay: 200 camas cada hora')
    ordered=stable_schedule.sort_values(['bed','hour']).copy()
    ordered['replacement']=ordered.groupby('bed').patient_id.transform(lambda s:s.ne(s.shift()).astype(int))
    counts=ordered.groupby('day').replacement.sum()
    ax[1].bar(counts.index,counts.values,color='#55a39a')
    ax[1].set(xlabel='Dia simulado',ylabel='Pacientes ingresados por dia',title='Ingresos observados (dia 1 incluye 200 camas iniciales)')
    save(fig,'01_ocupacion_recambio')

    fig,ax=plt.subplots(figsize=(16,9))
    for j,name in enumerate(SCENARIOS):
        if DAYS>=5 and name!='stable':ax.barh(j,DAYS-4,left=5,color=('#b9bec7','#5691ad','#bd8e49','#9b5971')[j])
    ax.set(yticks=range(4),yticklabels=SCENARIOS,xlim=(1,17),xlabel='Dia simulado',
           title='Calendario final: cambio estadistico desde dia 5')
    if DAYS>=5:ax.axvline(5,color='black',ls='--',lw=1)
    save(fig,'02_calendario_cambios')

    chosen=daily.loc[daily.policy.eq('dqn') & daily.scenario.eq('combined')].sort_values('day')
    fig,axes=plt.subplots(3,1,figsize=(16,9),sharex=True)
    for axis,col,thr in zip(axes,('ks','psi','auc'),(.30,.25,.85)):
        axis.plot(chosen.day,chosen[col],marker='o',color='#245c78',label=col.upper())
        axis.axhline(thr,ls='--',color='#a43b3b',label=f'Umbral {thr:.2f}')
        if DAYS>=5:axis.axvspan(5,DAYS,alpha=.08,color='#bd8e49')
        if col=='psi':axis.set_yscale('symlog',linthresh=.25)
        axis.set_ylabel(col.upper());axis.legend(loc='best')
    axes[0].set_title('Monitoreo diario antes de accion: escenario combinado')
    axes[-1].set_xlabel('Dia simulado')
    save(fig,'03_monitoreo_ks_psi_auc')

    hist=pd.DataFrame(train_history)
    fig,axes=plt.subplots(2,1,figsize=(16,9),sharex=True)
    axes[0].plot(hist.episode,hist.train_return,alpha=.3,label='Retorno entrenamiento crudo')
    axes[0].plot(hist.episode,hist.train_return.rolling(min(20,len(hist)),min_periods=1).mean(),label='Media movil 20')
    valid=hist.dropna(subset=['validation_return'])
    axes[0].plot(valid.episode,valid.validation_return,'o-',label='Validacion greedy')
    axes[0].set(ylabel='Retorno episodico',title='Aprendizaje del DQN; validacion sin exploracion')
    axes[0].legend()
    axes[1].plot(hist.episode,hist.epsilon,color='#bd8e49')
    axes[1].set(xlabel='Episodio',ylabel='Epsilon')
    save(fig,'04_entrenamiento_retorno_epsilon')

    plot_daily=daily.loc[daily.scenario.eq('combined')].sort_values('day')
    fig,axes=plt.subplots(2,1,figsize=(16,9),sharex=True)
    for policy in ('dqn','A1','A4','random','heuristic'):
        series=plot_daily.loc[plot_daily.policy.eq(policy)]
        axes[0].plot(series.day,series.auc,marker='.',label=policy)
        axes[1].plot(series.day,series.reward.fillna(0).cumsum(),marker='.',label=policy)
    axes[0].set(ylabel='AUC dia actual',title='AUC sobre pacientes reservados: escenario combinado')
    axes[1].set(xlabel='Dia simulado',ylabel='Reward acumulado (AUC - 0,1 x s)',
                title='Retorno acumulado: mismo replay para todas las politicas')
    axes[0].legend(ncol=5,loc='best')
    save(fig,'05_auc_y_reward_politicas')

    fig,ax=plt.subplots(figsize=(16,9))
    colors=['#879ba5','#478da7','#c18d48','#a95a79']
    for j,name in enumerate(SCENARIOS):
        data=actions.loc[actions.policy.eq('dqn') & actions.scenario.eq(name)].sort_values('day')
        ax.scatter(data.day, np.full(len(data),j),c=[colors[int(a)] for a in data.executed],s=130,marker='s')
    for i,name in enumerate(ACTIONS):ax.scatter([],[],c=colors[i],label=name,s=100,marker='s')
    ax.axvline(5,color='black',ls='--',lw=1)
    ax.set(xlabel='Dia de decision',yticks=range(4),yticklabels=SCENARIOS,title='Acciones ejecutadas por DQN en prueba reservada',xlim=(.5,DAYS))
    ax.legend(title='Accion',ncol=4)
    save(fig,'06_acciones_dqn')

    subset=audit.loc[audit.scenario.eq('combined')].sort_values('day')
    fig,axes=plt.subplots(2,1,figsize=(16,9),sharex=True)
    for i,name in enumerate(ACTIONS):axes[0].plot(subset.day,subset[f'utility_{name}'],marker='.',label=name,color=colors[i])
    axes[0].plot(subset.day,subset.chosen_utility,'ko',ms=5,label='Elegida')
    axes[0].set(ylabel='AUC siguiente - 0,1 x s',title='Oracle inmediato: cuatro acciones desde mismo pipeline')
    axes[0].legend(ncol=5)
    axes[1].bar(subset.day,subset.regret,color='#a95a79')
    axes[1].set(xlabel='Dia de decision',ylabel='Regret inmediato',title='Diferencia frente a mejor utilidad del dia')
    save(fig,'07_utilidad_acciones_regret')

    fig,axes=plt.subplots(2,1,figsize=(16,9),sharex=True)
    for name in ('covariate','combined'):
        data=daily.loc[daily.policy.eq('dqn') & daily.scenario.eq(name)].sort_values('day')
        axes[0].plot(data.day,data.saturation,marker='o',label=name)
        axes[1].plot(data.day,data.auc,marker='o',label=name)
    axes[0].set(ylabel='Fraccion de features saturados',title='Clipping bajo cambio de medicion')
    axes[1].set(xlabel='Dia simulado',ylabel='AUC',title='Recuperacion predictiva observada')
    axes[0].legend()
    save(fig,'08_saturacion_y_recuperacion')
    return files


### Ejecución, persistencia parcial y comprobación honesta de criterios



In [ ]:
def run_experiment():
    """Calibrate, train, compare policies, audit decisions and render figures."""
    offset,scale,calibrated,grid=calibrate()
    checkpoint('calibrated',offset_iqr=offset,scale=scale,calibration_qualified=calibrated)
    print('Calibración B-train:',offset,scale,'criterio:',calibrated)
    agent,history,checkpoints,training=train_agent(offset,scale)
    checkpoint('trained',**training)
    test_base=make_replay(B_PARTS['test'],days=DAYS,seed=SEED)
    summary_rows=[];all_actions=[];all_daily=[];all_audit=[]
    for scenario in SCENARIOS:
        schedule=inject(test_base,scenario,offset,scale,onset=5 if DAYS>=5 else 2)
        for policy in ('dqn',*ACTIONS,'random','heuristic'):
            result,acts,days,aud=roll(schedule,policy,agent,audit=(policy=='dqn'))
            summary_rows.append(result);all_actions.append(acts);all_daily.append(days)
            if len(aud):all_audit.append(aud)
        pd.DataFrame(summary_rows).to_csv(OUT/'policy_comparison.csv',index=False)
        checkpoint('evaluating',completed_scenarios=scenario)
    comparison=pd.DataFrame(summary_rows)
    actions=pd.concat(all_actions,ignore_index=True)
    daily=pd.concat(all_daily,ignore_index=True)
    audit=pd.concat(all_audit,ignore_index=True)
    actions.to_csv(OUT/'actions.csv',index=False)
    daily.to_csv(OUT/'daily_metrics.csv',index=False)
    audit.to_csv(OUT/'action_audit.csv',index=False)
    checkpoint('plotting')
    files=make_figures(inject(test_base,'stable',offset,scale),history,daily,actions,audit,comparison)
    dqn=comparison.loc[comparison.policy.eq('dqn')].set_index('scenario')
    a1=comparison.loc[comparison.policy.eq('A1')].set_index('scenario')
    random_policy=comparison.loc[comparison.policy.eq('random')].set_index('scenario')
    inner=audit.loc[~audit.frontier & audit.regret.notna()]
    action_set=set(actions.loc[actions.policy.eq('dqn'),'executed'])
    gates={'all_four_actions_functional':set(actions.executed)=={0,1,2,3},
           'positive_dqn_return':bool((dqn['return']>0).all()),
           'dqn_above_a1_random_drift':bool(all(dqn.loc[s,'return']>max(a1.loc[s,'return'],random_policy.loc[s,'return']) for s in SCENARIOS[1:])),
           'oracle_near_rate_80pct':bool(len(inner)>0 and inner.regret.le(.01).mean()>=.8),
           'dqn_selected_four_actions':action_set=={0,1,2,3},
           'validation_converged':bool(np.isfinite(training['validation_variation']) and training['validation_variation']<=.05 and training['policy_stability']>=.95),
           'calibration_qualified':calibrated}
    checkpoint('done',status='complete' if all(gates.values()) else 'needs_improvement',gates=gates,
               dqn_action_counts={ACTIONS[i]:int(actions.loc[actions.policy.eq('dqn'),'executed'].eq(i).sum()) for i in range(4)},
               oracle_near_rate=float(inner.regret.le(.01).mean()) if len(inner) else None,
               plots=files,files=['training_history.csv','daily_metrics.csv','actions.csv','action_audit.csv','policy_comparison.csv','calibration.csv'],
               training=training)
    print('Estado:',progress['status'],'| gates:',gates)
    print(comparison[['scenario','policy','mean_auc','return','seconds']].round(3).to_string(index=False))

try:
    run_experiment()
except Exception as exc:
    checkpoint('failed',status='failed',error=repr(exc),traceback=traceback.format_exc())
    print('Experimento falló:',repr(exc))
